# GPU Summarization Server

Runs heavy summarization models on Colab GPU and exposes them via ngrok.

**Models:**
- `led-large-book` — LED-Large (16K token input) fine-tuned on BookSum. English long documents.
- `flan-t5-large` — Google Flan-T5-Large with instruction-based summarization. English.
- `turkish-mbart` — mBART-Large fine-tuned on Turkish MLSum. Turkish summarization.

**Setup:** Runtime > Change runtime type > T4 GPU, then run all cells.

In [ ]:
# Cell 1: Install everything
!pip install flask pyngrok transformers torch sentencepiece protobuf accelerate

In [ ]:
# Cell 2: Check GPU
import torch
assert torch.cuda.is_available(), "NO GPU! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: Load models on GPU
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    LEDTokenizer,
    LEDForConditionalGeneration,
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
)
import torch, gc

# --- Model 1: LED-Large-Book-Summary (English, 16K context) ---
print("Loading LED-Large-Book-Summary...")
led_tokenizer = LEDTokenizer.from_pretrained("pszemraj/led-large-book-summary")
led_model = LEDForConditionalGeneration.from_pretrained("pszemraj/led-large-book-summary").to("cuda")
print(f"  Done. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# --- Model 2: Flan-T5-Large (English, instruction-tuned) ---
print("Loading Flan-T5-Large...")
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to("cuda")
print(f"  Done. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# --- Model 3: mBART-Large Turkish Summarization ---
print("Loading mBART-Large Turkish Summarization...")
tr_tokenizer = MBart50TokenizerFast.from_pretrained("mukayese/mbart-large-turkish-summarization")
tr_model = MBartForConditionalGeneration.from_pretrained("mukayese/mbart-large-turkish-summarization").to("cuda")
print(f"  Done. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

gc.collect()
torch.cuda.empty_cache()
print(f"\nAll models loaded! Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# Cell 4: Summarization functions with chunking
import torch, re

def split_into_chunks(text, tokenizer, max_tokens, overlap_sentences=2):
    """Split text into chunks that fit within max_tokens."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s.strip()]
    chunks = []
    current_chunk = []
    current_len = 0
    for sentence in sentences:
        sentence_tokens = len(tokenizer.encode(sentence, add_special_tokens=False))
        if current_len + sentence_tokens > max_tokens and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_len = sum(len(tokenizer.encode(s, add_special_tokens=False)) for s in current_chunk)
        current_chunk.append(sentence)
        current_len += sentence_tokens
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    return chunks if chunks else [text]


def summarize_led_single(text, max_length=300, min_length=50):
    """Summarize a single chunk with LED-Large."""
    inputs = led_tokenizer(text, return_tensors='pt', max_length=16384, truncation=True, padding='max_length').to('cuda')
    global_attention_mask = torch.zeros_like(inputs['input_ids'])
    global_attention_mask[:, 0] = 1
    with torch.no_grad():
        output = led_model.generate(
            **inputs, global_attention_mask=global_attention_mask,
            max_length=max_length, min_length=min(min_length, max_length - 10),
            num_beams=4, length_penalty=2.0, no_repeat_ngram_size=3,
        )
    return led_tokenizer.decode(output[0], skip_special_tokens=True)


def summarize_flan_single(text, max_length=200, min_length=50):
    """Summarize a single chunk with Flan-T5-Large."""
    prompt = f'Summarize the following text concisely, keeping only the most important facts and details:\n\n{text}'
    inputs = flan_tokenizer(prompt, return_tensors='pt', max_length=1024, truncation=True).to('cuda')
    with torch.no_grad():
        output = flan_model.generate(
            **inputs, max_length=max_length, min_length=min(min_length, max_length - 10),
            num_beams=4, length_penalty=1.0, no_repeat_ngram_size=3,
            repetition_penalty=2.5,
        )
    return flan_tokenizer.decode(output[0], skip_special_tokens=True)


def summarize_turkish_single(text, max_length=256, min_length=50):
    """Summarize a single chunk with mBART Turkish."""
    tr_tokenizer.src_lang = 'tr_TR'
    inputs = tr_tokenizer(text, return_tensors='pt', max_length=1024, truncation=True).to('cuda')
    with torch.no_grad():
        output = tr_model.generate(
            **inputs, max_length=max_length, min_length=min(min_length, max_length - 10),
            num_beams=4, length_penalty=1.0, no_repeat_ngram_size=3,
            forced_bos_token_id=tr_tokenizer.lang_code_to_id['tr_TR'],
        )
    return tr_tokenizer.decode(output[0], skip_special_tokens=True)


def summarize_with_chunking(text, model_name, max_length=300, min_length=50):
    """Hierarchical chunking: chunk > summarize each > merge > final summarize."""
    if model_name == 'led-large-book':
        tokenizer, chunk_limit, fn = led_tokenizer, 14000, summarize_led_single
    elif model_name == 'flan-t5-large':
        tokenizer, chunk_limit, fn = flan_tokenizer, 800, summarize_flan_single
    elif model_name == 'turkish-mbart':
        tokenizer, chunk_limit, fn = tr_tokenizer, 800, summarize_turkish_single
    else:
        return 'Unknown model'

    token_count = len(tokenizer.encode(text, add_special_tokens=False))
    if token_count <= chunk_limit:
        return fn(text, max_length=max_length, min_length=min_length)

    # Chunk, summarize each, merge, final pass
    chunks = split_into_chunks(text, tokenizer, chunk_limit, overlap_sentences=2)
    print(f'  [{model_name}] Split into {len(chunks)} chunks')
    chunk_max = max(80, max_length // len(chunks))
    chunk_min = max(20, chunk_max // 3)
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        chunk_summaries.append(fn(chunk, max_length=chunk_max, min_length=chunk_min))
        print(f'    Chunk {i+1}/{len(chunks)} done')
    merged = ' '.join(chunk_summaries)
    # Final pass
    merged_tokens = len(tokenizer.encode(merged, add_special_tokens=False))
    if merged_tokens > chunk_limit:
        return summarize_with_chunking(merged, model_name, max_length, min_length)
    print(f'  [{model_name}] Final summarization pass...')
    return fn(merged, max_length=max_length, min_length=min_length)


def summarize_led(text, max_length=300, min_length=50):
    return summarize_with_chunking(text, 'led-large-book', max_length, min_length)

def summarize_flan(text, max_length=200, min_length=50):
    return summarize_with_chunking(text, 'flan-t5-large', max_length, min_length)

def summarize_turkish(text, max_length=256, min_length=50):
    return summarize_with_chunking(text, 'turkish-mbart', max_length, min_length)

print('Summarization functions ready.')

In [ ]:
# Cell 5: Flask API server
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({
        'status': 'running',
        'gpu': torch.cuda.get_device_name(0),
        'models': ['led-large-book', 'flan-t5-large', 'turkish-mbart']
    })

@app.route('/summarize', methods=['POST'])
def summarize():
    data = request.get_json()
    text = data.get('text', '')
    model_name = data.get('model', 'led-large-book')
    max_length = data.get('max_length', 300)
    min_length = data.get('min_length', 50)

    if not text:
        return jsonify({'error': 'No text provided'}), 400

    try:
        if model_name == 'led-large-book':
            summary = summarize_led(text, max_length=max_length, min_length=min_length)
        elif model_name == 'flan-t5-large':
            summary = summarize_flan(text, max_length=max_length, min_length=min_length)
        elif model_name == 'turkish-mbart':
            summary = summarize_turkish(text, max_length=max_length, min_length=min_length)
        else:
            return jsonify({'error': f'Unknown model: {model_name}. Available: led-large-book, flan-t5-large, turkish-mbart'}), 400

        return jsonify({'summary': summary, 'model': model_name})
    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'traceback': traceback.format_exc()}), 500

@app.route('/summarize/batch', methods=['POST'])
def summarize_batch():
    data = request.get_json()
    text = data.get('text', '')
    max_length = data.get('max_length', 300)
    min_length = data.get('min_length', 50)
    if not text:
        return jsonify({'error': 'No text provided'}), 400
    results = {}
    for name, func in [('led-large-book', summarize_led), ('flan-t5-large', summarize_flan), ('turkish-mbart', summarize_turkish)]:
        try:
            results[name] = func(text, max_length=max_length, min_length=min_length)
        except Exception as e:
            results[name] = f'ERROR: {str(e)}'
    return jsonify({'results': results})

print('Flask app ready.')

In [ ]:
# Cell 6: Your ngrok token
# Get a free one at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_TOKEN_HERE"

In [ ]:
# Cell 7: Start server + ngrok tunnel
from pyngrok import ngrok
import threading

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)

threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

public_url = ngrok.connect(5000).public_url

print('=' * 60)
print(f'SERVER RUNNING AT: {public_url}')
print('=' * 60)
print(f'\nPut this in your .env file:')
print(f'COLAB_GPU_URL={public_url}')
print(f'\nTest: {public_url}/health')

In [ ]:
# Cell 8: Test all models
import requests, time
time.sleep(2)

print('Health:', requests.get(f'{public_url}/health').json())

en_text = 'Artificial intelligence has rapidly transformed from a theoretical concept to one of the most consequential technologies. The field was established at the Dartmouth Conference in 1956. Modern AI rests on neural networks trained via backpropagation. Large language models can write essays and generate code. Computer vision identifies objects with superhuman accuracy. Healthcare benefits through disease prediction and drug discovery. AI accelerates scientific research across disciplines.'

tr_text = 'Yapay zeka bilgisayar bilimlerinde teorik bir kavramdan onemli bir teknolojiye donustu. Modern yapay zekanin temeli sinir aglarina dayanmaktadir. Buyuk dil modelleri dogal dil anlama ve uretiminde olaganustu yetenekler gostermistir. Saglik sektoru yapay zekadan buyuk olcude faydalanacaktir. Bilimsel arastirmada yapay zeka bircok disiplinde kesfi hizlandirmaktadir.'

print('\n--- LED-Large (English) ---')
r = requests.post(f'{public_url}/summarize', json={'text': en_text, 'model': 'led-large-book', 'max_length': 100, 'min_length': 30})
print(r.json())

print('\n--- Flan-T5 (English) ---')
r = requests.post(f'{public_url}/summarize', json={'text': en_text, 'model': 'flan-t5-large', 'max_length': 100, 'min_length': 30})
print(r.json())

print('\n--- Turkish mBART (Turkish) ---')
r = requests.post(f'{public_url}/summarize', json={'text': tr_text, 'model': 'turkish-mbart', 'max_length': 100, 'min_length': 30})
print(r.json())

In [ ]:
# Cell 9: Keep alive
import time
print(f'Server: {public_url}')
print('Running... (stop this cell to shut down)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)